结构化输出是让llm不在返回难以解析的自由文本，而是直接输出符合你预期的schema


方式1：在prompt中规定返回json格式
方式2：with_structured_output()

In [1]:
from typing import TypedDict

from langchain.chat_models import init_chat_model
import os 
import traceback
from dotenv import load_dotenv

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

model = init_chat_model(
    model="glm-5.3-flash",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

response = model.invoke("给我推荐3本创业相关的书籍,返回json类型的格式,包含字段：{name,price,author,reason}")

print(response.content)


```json
{
  "books": [
    {
      "name": "《从0到1》",
      "price": "49元",
      "author": "彼得·蒂尔（Peter Thiel）",
      "reason": "PayPal创始人分享的创业思维经典之作，强调创新和垄断的重要性，教你如何创造全新的事物而非简单复制已有模式，适合所有想从零开始创业的人。"
    },
    {
      "name": "《精益创业》",
      "price": "59元",
      "author": "埃里克·莱斯（Eric Ries）",
      "reason": "提出了MVP（最小可行产品）、快速迭代等核心方法论，帮助创业者用最低成本验证商业模式，大幅降低创业失败风险，是创业方法论领域的必读书。"
    },
    {
      "name": "《创业维艰》",
      "price": "58元",
      "author": "本·霍洛维茨（Ben Horowitz）",
      "reason": "硅谷资深创业者 based on 20余年实战经验，直击创业中最艰难的时刻——裁员、融资、竞争、生死抉择，与那些讲'如何成功'的书不同，它告诉你'如何熬过难关'。"
    }
  ]
}
```

如果你想了解特定方向（如互联网创业、餐饮创业、商业模式设计等），我可以推荐更有针对性的书籍。


In [52]:
from pydantic import BaseModel,Field
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os 
load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


"""
    这段代码报错 ...
"""


# 定义结构化输出结构
class BookInfo(BaseModel):
    name :str = Field(description="书籍名称")
    price:float = Field(description="书籍价格，仅返回数字，单位元")
    author:str = Field(description="作者")
    reason:str = Field(description="推荐原因")

class BookList(BaseModel):
    books:list[BookInfo] = Field(description="书籍列表")


llm = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="openai",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    timeout=300
    # extra_body={"thinking": {"type": "disabled"}}
)


r = llm.invoke("你好")
print(r)

# 绑定结构化的输出
# structured_llm = llm.with_structured_output(BookInfo,method="json_mode")

# books = structured_llm.invoke("给出《创业维艰》的信息")

# print(books)




OpenAIConnectionError: Connection error.

In [64]:
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# 1. 定义 Schema
class BookInfo(BaseModel):
    name: str = Field(description="书籍名称")
    price: float = Field(description="书籍价格，仅返回数字，单位元")
    author: str = Field(description="作者")
    reason: str = Field(description="推荐原因")

class BookList(BaseModel):
    books: list[BookInfo] = Field(description="书籍列表")

# 2. 初始化模型时，关闭思考模式
llm = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
    reasoning_effort="none", # 关键：关闭思考模式
)

# 3. 使用 json_mode 策略
structured_llm = llm.with_structured_output(BookList, method="json_mode")

# 4. 在提示词中显式描述 JSON 结构（json_mode 必须手动指定）
prompt = """
请推荐三本计算机相关的书籍。
你必须返回一个 JSON 对象，且只包含 "books" 字段。
"books" 是一个数组，每个元素包含 "name", "price", "author", "reason" 字段。

示例格式：
{
  "books": [
    {"name": "书名", "price": 价格, "author": "作者", "reason": "推荐原因"}
  ]
}

请直接输出 JSON，不要包含任何解释、标题或 Markdown 标记。
"""

result = structured_llm.invoke(prompt)
print(result)


books=[BookInfo(name='深入理解计算机系统', price=139.0, author="Randal E. Bryant, David R. O'Hallaron", reason='从程序员视角深入剖析计算机系统底层原理，是理解计算机软硬件协同工作的经典之作。'), BookInfo(name='算法导论', price=128.0, author='Thomas H. Cormen, Charles E. Leiserson, Ronald L. Rivest, Clifford Stein', reason='算法领域的标准参考书，内容全面严谨，适合深入学习算法设计与分析。'), BookInfo(name='代码大全', price=98.0, author='Steve McConnell', reason='软件构建的实用百科全书，涵盖编码、调试、测试等最佳实践，提升代码质量与开发效率。')]


In [ ]:
# import os
# from langchain.chat_models import init_chat_model
# from pydantic import BaseModel, Field
# from dotenv import load_dotenv
# from langchain_deepseek import ChatDeepSeek   # 推荐用 ChatDeepSeek
# from langchain_core.output_parsers import PydanticOutputParser
# from langchain_core.prompts import ChatPromptTemplate

# load_dotenv(verbose=True)
# api_key = os.getenv("DEEPSEEK_API_KEY")
# print(api_key)
# print(base_url)

# class BookInfo(BaseModel):
#     name: str = Field(description="书籍名称")
#     price: float = Field(description="书籍价格，仅返回数字，单位元")
#     author: str = Field(description="作者")
#     reason: str = Field(description="推荐原因")

# class BookList(BaseModel):
#     books: list[BookInfo] = Field(description="书籍列表")

# # 1. 初始化模型（加超时）
# llm = init_chat_model(
#     model="deepseek-v4-pro",
#     model_provider="openai",
#     api_key=api_key,
#     base_url="https://api.deepseek.com/",
#     extra_body={"thinking": {"type": "disabled"}},  # 如需要可关闭思考模式
# )



# # 2. 先测试普通调用是否正常
# # r = llm.invoke("给我输出你好，其他不好输出")
# # print("普通调用成功：", r.content)


# structured_llm  = llm.with_structured_output(BookList,method="json_mode")

# structured_llm.invoke("给我推荐三本计算机相关的书籍")

# # 3. 结构化输出 —— 用 PydanticOutputParser（兼容性最好）


# # parser = PydanticOutputParser(pydantic_object=BookList)

# # prompt = ChatPromptTemplate.from_messages([
# #     ("system", "你是一个书籍推荐助手。请严格按以下格式输出，不要添加任何多余文字。\n{format_instructions}"),
# #     ("human", "给我推荐三本计算机相关的书籍")
# # ]).partial(format_instructions=parser.get_format_instructions())

# # chain = prompt | llm | parser
# # result = chain.invoke({})
# # print(result)
# # for book in result.books:
# #     print(f"{book.name} | {book.author} | {book.price}元")

sk-f366800c54ab4480beb5a9e7f21ee2a7
https://api.deepseek.com\


OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "Prompt must contain the word 'json' in some form to use 'response_format' of type 'json_object'. (request_id: ce1f348e-dc4e-4f66-bed1-9b70aab3ed18)", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}


TypedDict

使用TypedDict规定json有哪些属性和类型


In [7]:
from typing import TypedDict

class Book1(TypedDict):
    name:str
    price:float
    author:str
    reason:str

book1 = Book1(name="草房子",price=80,author="曹文轩",reason="推荐")
print(book1["name"])


# 字典字面量写法
book2 : Book1 = {
    "name": "水浒传",
    "price": 80,
    "author": "xx",
    "reason":"四大经典"
}

print(book2)





草房子
{'name': '水浒传', 'price': 80, 'author': 'xx', 'reason': '四大经典'}
